In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.metrics import mean_absolute_error, mean_squared_error
import os

# =========================
# SETTINGS
# =========================
INPUT_FILE = "../../DATA/environment_dataset.csv"
OUTPUT_DIR = "../../DATA/analysis_outputs/publication_sensors_output"
#os.makedirs(OUTPUT_DIR, exist_ok=True)

# Sensors
sensor_pairs = [
    ("temperature", "temperature_ref"),
    ("humidity", "humidity_ref"),
    ("co2", "co2_ref"),
    ("ammonia", "ammonia_ref")
]

df = pd.read_csv(INPUT_FILE)
df["timestamp"] = pd.to_datetime(df["timestamp"])

# =========================
# GLOBAL STYLE (journal-like)
# =========================
plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "legend.fontsize": 10
})

summary_results = []

# =========================
# LOOP THROUGH ALL SENSORS
# =========================
for sensor, ref in sensor_pairs:

    clean_df = df[[sensor, ref, "timestamp"]].dropna().copy()

    y_pred = clean_df[sensor].values
    y_true = clean_df[ref].values

    # =========================
    # ERROR METRICS
    # =========================
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    bias = np.mean(y_pred - y_true)

    # =========================
    # CORRELATION
    # =========================
    r, _ = stats.pearsonr(y_pred, y_true)

    # =========================
    # REGRESSION
    # =========================
    slope, intercept, r_val, _, _ = stats.linregress(y_true, y_pred)

    # =========================
    # BLAND-ALTMAN
    # =========================
    mean_pair = (y_pred + y_true) / 2
    diff = y_pred - y_true

    ba_bias = np.mean(diff)
    ba_sd = np.std(diff)
    loa_upper = ba_bias + 1.96 * ba_sd
    loa_lower = ba_bias - 1.96 * ba_sd

    # =========================
    # SAVE METRICS
    # =========================
    summary_results.append({
        "sensor": sensor,
        "MAE": mae,
        "RMSE": rmse,
        "Bias": bias,
        "Pearson_r": r,
        "R2": r_val**2,
        "BA_bias": ba_bias,
        "Upper_LoA": loa_upper,
        "Lower_LoA": loa_lower
    })

    # =========================
    # PLOTS
    # =========================

    # 📈 SCATTER
    plt.figure(figsize=(6.5, 5.5))
    plt.scatter(y_true, y_pred, s=18, alpha=0.6)

    x = np.linspace(min(y_true), max(y_true), 100)
    plt.plot(x, x, '--', label="Ideal")
    plt.plot(x, slope*x + intercept, label="Fit")

    plt.xlabel("Reference")
    plt.ylabel("Sensor")
    plt.title(f"{sensor.capitalize()} vs Reference")

    plt.text(0.05, 0.95, f"$R^2={r_val**2:.3f}$",
             transform=plt.gca().transAxes)

    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/{sensor}_scatter.png", dpi=300)
    plt.close()

    # 📉 BLAND-ALTMAN
    plt.figure(figsize=(6.5, 5.5))
    plt.scatter(mean_pair, diff, s=18, alpha=0.6)

    plt.axhline(ba_bias, linestyle="--", label="Bias")
    plt.axhline(loa_upper, linestyle="--", label="+1.96SD")
    plt.axhline(loa_lower, linestyle="--", label="-1.96SD")

    plt.xlabel("Mean")
    plt.ylabel("Difference")
    plt.title(f"Bland-Altman: {sensor}")

    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/{sensor}_bland_altman.png", dpi=300)
    plt.close()

    # 📊 ERROR HISTOGRAM
    plt.figure(figsize=(6.5, 5.5))
    plt.hist(diff, bins=30, edgecolor='black')

    plt.axvline(ba_bias, linestyle="--")

    plt.xlabel("Error")
    plt.ylabel("Frequency")
    plt.title(f"Error Distribution: {sensor}")

    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/{sensor}_error.png", dpi=300)
    plt.close()

# =========================
# SAVE SUMMARY TABLE
# =========================
summary_df = pd.DataFrame(summary_results)
summary_df.to_csv(f"{OUTPUT_DIR}/summary_results.csv", index=False)

print("All publication plots and tables generated.")

In [ ]:
import os
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.metrics import mean_absolute_error, mean_squared_error

# =========================
# SETTINGS
# =========================
INPUT_FILE = "../../DATA/environment_dataset.csv"   # change if needed
OUTPUT_DIR = "../../DATA/analysis_outputs/publication_sensors_output/latex_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Candidate sensor-reference pairs
candidate_pairs = [
    ("temperature", "temperature_ref", "Temperature"),
    ("humidity", "humidity_ref", "Humidity"),
    ("co2", "co2_ref", "CO$_2$"),
    ("ammonia", "ammonia_ref", "Ammonia"),
    ("weight", "weight_ref", "Weight"),
    ("raw_weight_kg", "raw_weight_kg_ref", "Weight"),
]

# =========================
# LOAD
# =========================
df = pd.read_csv(INPUT_FILE)

if "timestamp" in df.columns:
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")

# Keep only available pairs
sensor_pairs = []
seen_labels = set()
for sensor, ref, label in candidate_pairs:
    if sensor in df.columns and ref in df.columns:
        # avoid duplicate weight entry if both naming styles exist
        if label not in seen_labels:
            sensor_pairs.append((sensor, ref, label))
            seen_labels.add(label)

if not sensor_pairs:
    raise ValueError("No valid sensor/reference column pairs found in the CSV.")

# =========================
# HELPERS
# =========================
def safe_mape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = y_true != 0
    if mask.sum() == 0:
        return np.nan
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def icc2_1(data):
    data = np.asarray(data, dtype=float)
    n, k = data.shape

    mean_target = np.mean(data, axis=1, keepdims=True)
    mean_rater = np.mean(data, axis=0, keepdims=True)
    grand_mean = np.mean(data)

    ss_total = np.sum((data - grand_mean) ** 2)
    ss_targets = k * np.sum((mean_target - grand_mean) ** 2)
    ss_raters = n * np.sum((mean_rater - grand_mean) ** 2)
    ss_error = ss_total - ss_targets - ss_raters

    df_targets = n - 1
    df_raters = k - 1
    df_error = (n - 1) * (k - 1)

    ms_targets = ss_targets / df_targets
    ms_raters = ss_raters / df_raters if df_raters != 0 else 0
    ms_error = ss_error / df_error if df_error != 0 else 0

    return (ms_targets - ms_error) / (
        ms_targets + (k - 1) * ms_error + (k * (ms_raters - ms_error) / n)
    )

def to_latex_table(df_table, caption, label, column_format=None):
    return df_table.to_latex(
        index=False,
        escape=False,
        caption=caption,
        label=label,
        column_format=column_format
    )

# =========================
# ANALYSIS
# =========================
desc_rows = []
performance_rows = []
agreement_rows = []

for sensor, ref, label in sensor_pairs:
    sub = df[[sensor, ref]].copy()
    sub[sensor] = pd.to_numeric(sub[sensor], errors="coerce")
    sub[ref] = pd.to_numeric(sub[ref], errors="coerce")
    sub = sub.dropna().reset_index(drop=True)

    if len(sub) < 3:
        continue

    y_pred = sub[sensor].values
    y_true = sub[ref].values

    # Descriptive
    desc_rows.append({
        "Variable": label,
        "n": len(sub),
        "Sensor Mean": np.mean(y_pred),
        "Sensor SD": np.std(y_pred, ddof=1),
        "Reference Mean": np.mean(y_true),
        "Reference SD": np.std(y_true, ddof=1),
        "Sensor Min": np.min(y_pred),
        "Sensor Max": np.max(y_pred),
        "Reference Min": np.min(y_true),
        "Reference Max": np.max(y_true),
    })

    # Performance
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    bias = np.mean(y_pred - y_true)
    mape = safe_mape(y_true, y_pred)

    pearson_r, pearson_p = stats.pearsonr(y_pred, y_true)
    slope, intercept, r_val, reg_p, std_err = stats.linregress(y_true, y_pred)

    performance_rows.append({
        "Variable": label,
        "MAE": mae,
        "RMSE": rmse,
        "Bias": bias,
        "MAPE (\\%)": mape,
        "Pearson $r$": pearson_r,
        "$p$-value": pearson_p,
        "Slope": slope,
        "Intercept": intercept,
        "$R^2$": r_val**2,
    })

    # Agreement
    diff = y_pred - y_true
    mean_pair = (y_pred + y_true) / 2

    ba_bias = np.mean(diff)
    ba_sd = np.std(diff, ddof=1)
    loa_upper = ba_bias + 1.96 * ba_sd
    loa_lower = ba_bias - 1.96 * ba_sd
    pct_outside = np.mean((diff > loa_upper) | (diff < loa_lower)) * 100

    try:
        shapiro_stat, shapiro_p = stats.shapiro(diff[:5000] if len(diff) > 5000 else diff)
    except Exception:
        shapiro_stat, shapiro_p = np.nan, np.nan

    try:
        icc_val = icc2_1(sub[[sensor, ref]].values)
    except Exception:
        icc_val = np.nan

    agreement_rows.append({
        "Variable": label,
        "BA Bias": ba_bias,
        "BA SD": ba_sd,
        "Upper LoA": loa_upper,
        "Lower LoA": loa_lower,
        "Outside LoA (\\%)": pct_outside,
        "ICC(2,1)": icc_val,
        "Shapiro $p$": shapiro_p,
    })

# =========================
# DATAFRAMES
# =========================
desc_df = pd.DataFrame(desc_rows)
perf_df = pd.DataFrame(performance_rows)
agree_df = pd.DataFrame(agreement_rows)

# Round for publication
desc_df = desc_df.round(3)
perf_df = perf_df.round(4)
agree_df = agree_df.round(4)

# Save CSV too
desc_df.to_csv(os.path.join(OUTPUT_DIR, "descriptive_statistics.csv"), index=False)
perf_df.to_csv(os.path.join(OUTPUT_DIR, "performance_metrics.csv"), index=False)
agree_df.to_csv(os.path.join(OUTPUT_DIR, "agreement_metrics.csv"), index=False)

# =========================
# LATEX TABLES
# =========================
latex_desc = to_latex_table(
    desc_df,
    caption="Descriptive statistics for sensor and reference measurements across all monitored variables.",
    label="tab:descriptive_stats",
    column_format="lrrrrrrrrr"
)

latex_perf = to_latex_table(
    perf_df,
    caption="Performance metrics comparing sensor outputs against reference measurements.",
    label="tab:performance_metrics",
    column_format="lrrrrrrrrr"
)

latex_agree = to_latex_table(
    agree_df,
    caption="Agreement analysis based on Bland--Altman statistics and ICC for all monitored variables.",
    label="tab:agreement_metrics",
    column_format="lrrrrrrrr"
)

# Combined compact summary table
summary_df = perf_df[["Variable", "MAE", "RMSE", "Bias", "Pearson $r$", "$R^2$"]].merge(
    agree_df[["Variable", "BA Bias", "Upper LoA", "Lower LoA", "ICC(2,1)"]],
    on="Variable",
    how="inner"
).round(4)

latex_summary = to_latex_table(
    summary_df,
    caption="Compact summary of validation results across all variables.",
    label="tab:validation_summary",
    column_format="lrrrrrrrrr"
)

# Write individual tex files
with open(os.path.join(OUTPUT_DIR, "table_descriptive_stats.tex"), "w", encoding="utf-8") as f:
    f.write(latex_desc)

with open(os.path.join(OUTPUT_DIR, "table_performance_metrics.tex"), "w", encoding="utf-8") as f:
    f.write(latex_perf)

with open(os.path.join(OUTPUT_DIR, "table_agreement_metrics.tex"), "w", encoding="utf-8") as f:
    f.write(latex_agree)

with open(os.path.join(OUTPUT_DIR, "table_validation_summary.tex"), "w", encoding="utf-8") as f:
    f.write(latex_summary)

# Also write a single file containing all tables
all_tables = "\n\n".join([
    latex_desc,
    latex_perf,
    latex_agree,
    latex_summary
])

with open(os.path.join(OUTPUT_DIR, "all_tables.tex"), "w", encoding="utf-8") as f:
    f.write(all_tables)

print("LaTeX tables generated in:", OUTPUT_DIR)
print("Files:")
print("- table_descriptive_stats.tex")
print("- table_performance_metrics.tex")
print("- table_agreement_metrics.tex")
print("- table_validation_summary.tex")
print("- all_tables.tex")